# 03 · Model evaluation

Loads the metrics written by `python -m scripts.run_experiment` and puts the ML models and the baselines side by side.
No model is trained in this notebook. Every number comes from `results/metrics/`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.utils import load_config
cfg = load_config(ROOT / "config.yaml")
RESULTS = ROOT / cfg["experiment"]["results_dir"]

In [ ]:
import json
summary = json.loads((RESULTS / "metrics" / "run_summary.json").read_text())
print("synthetic data:", summary["synthetic_data"]); print(summary["environment"])

## Classification (test period)
Compare each model with the **majority-class** row: beating it is the minimum bar for useful predictions.

In [ ]:
cls = pd.read_csv(RESULTS / "metrics" / "classification_metrics.csv")
cls[cls.split == "test"][["ticker", "model", "accuracy", "balanced_accuracy", "precision_macro", "recall_macro", "f1_macro", "log_loss", "brier"]].round(4)

## Trading (test period, net of costs)

In [ ]:
comp = pd.read_csv(RESULTS / "metrics" / "model_comparison_test.csv")
comp[comp.kind.isin(["baseline", "ml_confidence"])][["ticker", "strategy", "threshold", "cumulative_return", "sharpe", "sortino", "max_drawdown", "turnover_annual", "n_trades"]].round(3)

## How the confidence threshold was chosen (validation period only)

In [ ]:
t = cfg["data"]["tickers"][0]
pd.read_csv(RESULTS / "metrics" / "tuning" / f"{t}_xgboost_threshold_selection_val.csv").round(3)

## Hyperparameter search (purged time-series CV on the training period)

In [ ]:
pd.read_csv(RESULTS / "metrics" / "tuning" / f"{t}_xgboost_search.csv").head(10).round(4)

## Statistical uncertainty: block-bootstrap 95% intervals

In [ ]:
pd.read_csv(RESULTS / "metrics" / "bootstrap_confidence_intervals.csv").round(3)

## Walk-forward (annual retraining)

In [ ]:
wf_path = RESULTS / "metrics" / "walk_forward_trading.csv"
if wf_path.exists():
    display(pd.read_csv(wf_path)[["ticker", "strategy", "start", "end", "cumulative_return", "sharpe", "max_drawdown"]].round(3))
    display(pd.read_csv(RESULTS / "metrics" / "walk_forward_classification.csv").round(4))

## Key figures

In [ ]:
for name in ["classification_comparison.png", "sharpe_comparison.png", f"{t}_equity_curves.png", f"{t}_confusion_matrices.png",
             f"{t}_reliability.png", "cost_sensitivity.png", "walk_forward_sharpe.png"]:
    p = RESULTS / "figures" / name
    if p.exists():
        display(Markdown(f"**{name}**")); display(Image(filename=str(p)))